# Dataset Notebook

- Source: `src/dataset.py`
- 목적: 원본 파이썬 파일을 단계별로 실행/설명하기 위한 노트북 버전
- 실행 방법: 위에서 아래로 순서대로 실행


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # 노트북이 다른 경로에서 열렸을 때 프로젝트 루트 자동 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'src').exists() and (parent / 'configs').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


## Step 1. Setup and Imports

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
"""Single-task dataset + Albumentations transforms (Kaggle 전용).

이 모듈이 하는 일
─────────────────
1. get_transforms  : task 종류(roast/defect)와 train/val 여부에 따라
                     Albumentations 증강 파이프라인을 만들어 반환한다.
2. SingleTaskDataset : CSV 파일을 읽어 (이미지, 라벨) 쌍을 내주는 PyTorch Dataset.
3. compute_class_weights : 클래스 불균형을 보정하기 위한 역빈도 가중치를 계산한다.
"""
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2


## Step 2. Function: get_transforms

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def get_transforms(task: str = "roast", train: bool = True, img_size: int = 224):
    """task별 이미지 증강 파이프라인을 반환한다.

    왜 task마다 다르게?
    - roast  : 콩 색깔(초록→연갈→갈→검정)이 정답과 직결된다.
               색을 크게 바꾸는 증강(Hue, Saturation 등)을 넣으면 오히려 노이즈가 된다.
    - defect : 깨짐·곰팡이·표면 질감 같은 '형태' 정보가 중요하다.
               ColorJitter나 GaussianBlur를 넣어도 정보 손실이 적다.

    train=False (검증/테스트/추론)
    - 결과를 매번 다르게 만드는 랜덤 증강은 넣지 않는다.
    - Resize + Normalize만 적용해서 재현 가능한 결과를 보장한다.
    """
    if not train:
        # 검증/추론 때는 결과를 흔들지 않도록 resize + normalize만 적용한다.
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(),   # ImageNet 평균/표준편차로 정규화 (pretrained 모델 기준)
            ToTensorV2(),    # HWC numpy → CHW torch.Tensor 변환
        ])

    if task == "roast":
        # 로스팅 분류는 색감 변화가 정답에 직접 연결되므로
        # 밝기/대비 정도만 가볍게 흔들고 과한 색 왜곡은 피한다.
        return A.Compose([
            # scale=(0.85,1.0) : 원본의 85~100% 크기 영역을 잘라 img_size로 리사이즈
            A.RandomResizedCrop(size=(img_size, img_size), scale=(0.85, 1.0)),
            A.HorizontalFlip(p=0.5),   # 좌우 반전 (방향 무관)
            A.VerticalFlip(p=0.5),     # 상하 반전 (방향 무관)
            A.Rotate(limit=20, p=0.5), # 최대 ±20도 회전
            # brightness_limit=0.1, contrast_limit=0.1 : 약한 밝기·대비 흔들기
            A.RandomBrightnessContrast(0.1, 0.1, p=0.5),
            A.Normalize(),
            ToTensorV2(),
        ])

    # defect: 결점두 분류는 색뿐 아니라 모양, 표면 질감 차이도 중요하므로
    # roast보다 조금 더 강한 증강을 허용한다.
    return A.Compose([
        # scale=(0.8,1.0) : roast보다 더 넓은 crop 범위 허용
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=30, p=0.5),  # roast보다 회전 범위 더 허용
        # hue_shift_limit=0.05 : 색조 변화는 아주 약하게만 허용
        A.ColorJitter(0.2, 0.2, 0.2, 0.05, p=0.5),
        # 약간의 블러로 선명도 과의존 방지
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.Normalize(),
        ToTensorV2(),
    ])


## Step 3. Class: SingleTaskDataset

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
class SingleTaskDataset(Dataset):
    """CSV 파일 한 줄 = 이미지 한 장으로 취급하는 단일 태스크 Dataset.

    CSV 필수 컬럼
    - path        : 이미지 절대(또는 상대) 경로
    - roast_label : task='roast'일 때 사용 ('green' | 'light' | 'medium' | 'dark')
    - defect_label: task='defect'일 때 사용 (17개 결점두 클래스 이름)

    class_to_idx를 외부에서 주입하는 이유
    - make_splits.py가 만든 CSV의 클래스 순서와
      config.yaml에 적힌 classes 리스트 순서를 일치시켜
      train / val / test / streamlit이 모두 같은 인덱스를 공유하기 위해서다.
    """


## Step 4. Function: __init__

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
    def __init__(
        self,
        csv_path: str | Path,
        task: str = "roast",
        class_to_idx: dict[str, int] | None = None,
        transform=None,
    ):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        # task 값에 따라 roast_label 또는 defect_label 컬럼을 읽게 된다.
        self.label_col = f"{task}_label"
        if self.label_col not in self.df.columns:
            raise KeyError(f"CSV에 '{self.label_col}' 컬럼이 없습니다.")
        self.classes = sorted(self.df[self.label_col].unique().tolist())
        # 외부에서 class_to_idx를 넘기면 학습/평가/추론 전체에서
        # 클래스 인덱스 순서를 동일하게 맞출 수 있다.
        self.class_to_idx = class_to_idx or {c: i for i, c in enumerate(self.classes)}
        self.transform = transform


## Step 5. Function: __len__

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
    def __len__(self) -> int:
        return len(self.df)


## Step 6. Function: __getitem__

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        # Albumentations는 HWC numpy 이미지를 기대하므로
        # PIL 이미지를 RGB numpy 배열로 먼저 바꾼다.
        img = np.array(Image.open(row["path"]).convert("RGB"))
        if self.transform:
            img = self.transform(image=img)["image"]
        # 최종 라벨은 CrossEntropyLoss에 맞게 정수 인덱스로 반환한다.
        y = self.class_to_idx[row[self.label_col]]
        return img, torch.tensor(y, dtype=torch.long)


## Step 7. Function: compute_class_weights

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def compute_class_weights(csv_path: str | Path, task: str,
                          class_to_idx: dict[str, int]) -> torch.Tensor:
    """클래스 불균형 보정을 위한 역빈도(inverse-frequency) 가중치를 반환한다.

    공식:  weight[c] = total / (n_classes × count[c])
    → 샘플이 적은 클래스일수록 weight가 커져서 loss에서 더 크게 반영된다.
    → CrossEntropyLoss(weight=...) 에 바로 넘길 수 있는 형태다.

    결점두 데이터처럼 클래스별 샘플 수 차이가 클 때 특히 효과적이다.
    (예: 'full black' 50장 vs 'immature' 800장 → full black weight가 더 커짐)
    """
    df = pd.read_csv(csv_path)
    counts = df[f"{task}_label"].value_counts().to_dict()
    total = sum(counts.values())
    n_cls = len(class_to_idx)
    weights = torch.ones(n_cls, dtype=torch.float)
    # total / (n_cls * class_count) 형태의 역빈도 가중치다.
    # 샘플 수가 적은 클래스일수록 loss에서 조금 더 크게 반영된다.
    for cls, idx in class_to_idx.items():
        c = counts.get(cls, 0)
        weights[idx] = total / (n_cls * c) if c > 0 else 0.0
    return weights
